## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model='groq:openai/gpt-oss-120b',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

**The capital of the Moon is Lunaris‑Prime**, a sprawling, dome‑clad metropolis perched on the rim of the Shackleton Crater at the lunar south pole.  

---

### Why Lunaris‑Prime?

| Feature | Description |
|---------|-------------|
| **Location** | The crater’s permanently shadowed floor holds vast deposits of water ice, while its rim enjoys near‑continuous sunlight. This duality makes it the perfect spot for a self‑sustaining city that can harvest both solar power and water. |
| **Political Role** | Lunaris‑Prime is the seat of the **Lunar Commonwealth Council**, the governing body that coordinates everything from terra‑formation projects to inter‑planetary trade. The Council’s Hall of Orbits is an ultra‑transparent crystal structure that offers a 360° view of the Earth rising over the horizon. |
| **Population** | Roughly 12 million residents: scientists, engineers, artists, miners, and the first generation of “Moonborn” humans who have never set foot on Earth. |
| **Economy** | The

## Image input

In [8]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [10]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x0000022AF10677C0>, 'last_modified': datetime.datetime(2026, 10, 3, 10, 33, 58, 646000, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': 'messages[1].content must be a string', 'type': 'invalid_request_error', 'param': 'messages[1].content'}}

## Audio input

In [17]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.86it/s]


Done.


In [18]:
# Note: Audio input requires an OpenAI audio model (e.g. 'gpt-4o-audio-preview').
# Groq does not currently support audio input.
# To run this cell, set OPENAI_API_KEY in your .env and use:
#   agent = create_agent(model='gpt-4o-audio-preview')
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}